# VECTORBT

In [ ]:
import importlib
import src.utils.db as db_module
import src.utils.store as store_module
import src.utils.theme as theme_module

importlib.reload(db_module)
importlib.reload(store_module)
importlib.reload(theme_module)

from typing import cast
from src.utils.db import Db
import vectorbt as vbt
from src.utils.store import Store
from src.utils.theme import vscode_dark
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np

db = Db("vectorbt/tutorial")

In [ ]:
dfs = Store[pd.DataFrame]()
inds = Store[vbt.indicators.IndicatorBase]()
signals = Store[any]()
pfs = Store[vbt.Portfolio]()
trs = Store[any]()
views = Store[any]()

In [ ]:
db.exec("types")
db.exec("create-table")
db.exec("procedure.insert.array")

In [ ]:
end_time = datetime.now()
start_time = end_time - timedelta(days=1)

dfs.latest = "BTC-USD-Close", pd.DataFrame(
    vbt.YFData.download(
        ["BTC-USD", "ETH-USD"],
        missing_index="drop",
        start=start_time,
        end=end_time,
        interval="1m",
    ).get("Close")
)
dfs.latest

In [ ]:
rsi = vbt.RSI.run(dfs.latest, window=14)
rsi

In [ ]:
print(rsi.rsi)

In [ ]:
entries = rsi.rsi_crossed_below(30).loc[:, (14)]
exits = rsi.rsi_crossed_above(70).loc[:, (14)]

with plt.style.context(vscode_dark):
    fig, ax = plt.subplots()
    rsi.rsi.plot(ax=ax)
    plt.show()

In [ ]:
pf = vbt.Portfolio.from_signals(store.latest, entries, exits)
pf

In [ ]:
pf.stats()

In [ ]:
pf.total_return()

In [ ]:
print(pf.order_records)

This next plot method breaks jupyter for some reason
I had to restart WSL several times

In [ ]:
# pf[pf.total_return().idxmax()].plot()

## Custom indicator

In [ ]:
# ANKI DONE
def custom_indicator(close: pd.Series, window: int):
    print(close)
    rsi = vbt.RSI.run(close, window)
    return rsi.rsi


inds.latest = "rsi", vbt.IndicatorFactory(
    class_name="Combination",
    short_name="Comb",
    input_names=["close"],
    param_names=["window"],
    output_names=["values"],
).from_apply_func(custom_indicator, window=14)

In [ ]:
# ANKI DONE
res = inds.latest.run(dfs.latest, window=21)
res.values

In [ ]:
# ANKI DONE
def rsi_ma_indicator(close: pd.Series, rsi_window: int, ma_window):
    rsi = vbt.RSI.run(close, window=rsi_window).rsi.to_numpy()
    ma = vbt.MA.run(close, window=ma_window).ma.to_numpy()
    trend = np.where((rsi > 70) & (close > ma), 1, 0)
    trend = np.where((rsi < 30) & (close < ma), -1, trend)
    return trend


inds.latest = "RSI MA", vbt.IndicatorFactory(
    class_name="RSI MA",
    short_name="RM",
    input_names=["close"],
    param_names=["rsi_window", "ma_window"],
    output_names=["values"],
).from_apply_func(
    rsi_ma_indicator,
    rsi_window=14,
    ma_window=50,
)

In [ ]:
# ANKI DONE
signals.latest = "RSI MA", inds.latest.run(dfs.latest).values
signals.latest

In [ ]:
# ANKI DONE
entries = signals.latest == 1.0
exits = signals.latest == -1.0

In [ ]:
# ANKI DONE
pfs.latest = "RSI MA", vbt.Portfolio.from_signals(dfs.latest, entries, exits)
pfs.latest

In [ ]:
pfs.latest.stats()

### Chunking rsi for 5 minutes

In [ ]:
def rsi5_ma_indicator(
    close_pd: pd.Series,
    rsi_window: int,
    rsi_low: int,
    rsi_high: int,
    ma_window: int,
):
    close_5m = close_pd.resample("5min").last()
    rsi_5m = vbt.RSI.run(close_5m, window=rsi_window).rsi
    rsi_pd = rsi_5m.align(close_pd, join="right", axis=0)[0].ffill()
    ma = vbt.MA.run(close_pd, ma_window).ma.to_numpy()
    close = close_pd.to_numpy()
    rsi = rsi_pd.to_numpy()
    trend = np.where(rsi > rsi_high, -1, 0)
    trend = np.where((rsi < rsi_low) & (close < ma), 1, trend)
    return trend


inds.latest = "RSI5_MA", vbt.IndicatorFactory(
    class_name="RSI5_MA",
    short_name="R5M",
    input_names=["close"],
    param_names=["rsi_window", "rsi_low", "rsi_high", "ma_window"],
    output_names=["values"],
).from_apply_func(
    rsi5_ma_indicator,
    rsi_window=14,
    rsi_low=30,
    rsi_high=70,
    ma_window=50,
    keep_pd=True,
)

In [ ]:
signals.latest = (
    "RSI5_MA",
    inds.latest.run(
        dfs.latest,
        rsi_window=14,
        rsi_low=30,
        rsi_high=70,
        ma_window=50,
    ).values,
)
entries = signals.latest == 1
exits = signals.latest == -1
pfs.latest = "RSI5_MA", vbt.Portfolio.from_signals(dfs.latest, entries, exits)
pfs.latest[pfs.latest.total_return().idxmax()].stats()

## Hyperparameter optimization

In [ ]:
signals.latest = (
    "RM_4d",
    inds.latest.run(
        dfs.latest,
        param_product=True,
        rsi_window=[14, 21, 30],
        ma_window=[21, 50, 100],
        rsi_low=[30, 40],
        rsi_high=[60, 70],
    ).values,
)
entries = signals.latest == 1
exits = signals.latest == -1
pfs.latest = "RM_4d", vbt.Portfolio.from_signals(dfs.latest, entries, exits)
pfs.latest[pfs.latest.total_return().idxmax()].stats()

In [ ]:
with pd.option_context(
    "display.expand_frame_repr", None, "display.max_rows", None
):
    print(pfs.latest.total_return())

In [ ]:
pfs.latest[pfs.latest.total_return().idxmax()].stats()

In [ ]:
# Just eth returns
tr = pfs.latest.total_return()
tr[tr.index.isin(["ETH-USD"], level="symbol")]

## Using ranges

In [ ]:
# ANKI DONE
signals.latest = (
    "RM_range",
    inds.latest.run(
        dfs.latest,
        param_product=True,
        rsi_window=np.arange(10, 40, step=3, dtype=int),
        ma_window=np.arange(10, 100, step=14, dtype=int),
        rsi_low=np.arange(10, 40, step=3, dtype=int),
        rsi_high=np.arange(60, 80, step=3, dtype=int),
    ).values,
)
entries = signals.latest == 1
exits = signals.latest == -1
pfs.latest = "RM_range", vbt.Portfolio.from_signals(dfs.latest, entries, exits)
pfs.latest[pfs.latest.total_return().idxmax()].stats()

In [ ]:
trs.latest = "RM_range", pfs.latest.total_return()
pfs.latest[trs.latest.idxmin()].stats()

## Plotting

In [ ]:
signals.latest = (
    "RM_low_high",
    inds.latest.run(
        dfs.latest,
        param_product=True,
        # rsi_window=np.arange(10, 40, step=3, dtype=int),
        # ma_window=np.arange(10, 100, step=14, dtype=int),
        rsi_low=np.arange(10, 40, step=3, dtype=int),
        rsi_high=np.arange(60, 80, step=3, dtype=int),
    ).values,
)
entries = signals.latest == 1
exits = signals.latest == -1
pfs.latest = "RM_low_high", vbt.Portfolio.from_signals(
    dfs.latest, entries, exits
)
trs.latest = "RM_low_high", pfs.latest.total_return()

In [ ]:
fig = trs.latest.vbt.heatmap(
    x_level="R5M_rsi_low",
    y_level="R5M_rsi_high",
    slider_level="symbol",
)
fig.show()

In [ ]:
signals.latest = (
    "RM_low_high",
    inds.latest.run(
        dfs.latest,
        param_product=True,
        rsi_window=np.arange(10, 40, step=3, dtype=int),
        # ma_window=np.arange(10, 100, step=14, dtype=int),
        rsi_low=np.arange(10, 40, step=3, dtype=int),
        rsi_high=np.arange(60, 80, step=3, dtype=int),
    ).values,
)
entries = signals.latest == 1
exits = signals.latest == -1
pfs.latest = "RM_low_high", vbt.Portfolio.from_signals(
    dfs.latest, entries, exits
)
trs.latest = "RM_low_high", pfs.latest.total_return()
fig = trs.latest.vbt.heatmap(
    x_level="R5M_rsi_low",
    y_level="R5M_rsi_high",
    slider_level="symbol",
)
fig.show()

In [ ]:
name = "RM_view"
signals.latest = (
    name,
    inds.latest.run(
        dfs.latest,
        param_product=True,
        rsi_window=np.arange(10, 40, step=3, dtype=int),
        ma_window=np.arange(10, 100, step=14, dtype=int),
        rsi_low=np.arange(10, 40, step=3, dtype=int),
        rsi_high=np.arange(60, 80, step=3, dtype=int),
    ).values,
)
entries = signals.latest == 1
exits = signals.latest == -1
pfs.latest = name, vbt.Portfolio.from_signals(dfs.latest, entries, exits)
trs.latest = name, pfs.latest.total_return()
views.latest = (
    name,
    trs.latest.groupby(level=["R5M_rsi_low", "R5M_rsi_high", "symbol"]).mean(),
)
views.latest
# fig = trs.latest.vbt.heatmap(
#     x_level="R5M_rsi_low",
#     y_level="R5M_rsi_high",
#     slider_level="symbol",
# )
# fig.show()

In [ ]:
fig = views.latest.vbt.heatmap(
    x_level="R5M_rsi_low",
    y_level="R5M_rsi_high",
    slider_level="symbol",
)
fig.show()